# Exercise 10: Average Time of Process per Machine

**Source:** [Leet Code 1661](https://leetcode.com/problems/average-time-of-process-per-machine/description/?envType=study-plan-v2&envId=top-sql-50) (Easy).

Table: `Activity`
```
+----------------+---------+
| Column Name    | Type    |
+----------------+---------+
| machine_id     | int     |
| process_id     | int     |
| activity_type  | enum    |
| timestamp      | float   |
+----------------+---------+
```
- `process_id` is the process number running on the machine with `machine_id`.
- `activity_type` is a categorical feature with `start` and `end` denoting whether it is the beginning or end of `process_id`.
- `timestamp` is a float marking a time measurement.

**Task.**
Find the average processing time `processing_time` for each machine **rounded to 3 decimal places** in **any order**

Connect to the database `postgres` using the username `postgres`.

In [1]:
%load_ext sql
%sql postgresql+psycopg://postgres@localhost:5432/postgres

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'postgresql+psycopg://postgres@localhost:5432/postgres'

### Create new database for the exercise

Make sure first that a database with the same name is removed in case it already exists.

In [2]:
%sql DROP DATABASE IF EXISTS exercise_10 WITH (FORCE);
%sql CREATE DATABASE exercise_10;

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

Connect to the database.

*Caution.* No semicolon at the end otherwise it belongs to the name of the database.

In [3]:
%sql postgresql+psycopg://postgres@localhost:5432/exercise_10 --alias exercise_10

Connecting and switching to connection 'exercise_10'

#### Fill the database with content

The `ENUM` feature for the `activity_type` needs to be created.

In [6]:
%sql CREATE TYPE status AS ENUM ('start', 'end');

Running query in 'exercise_10'

++
||
++
++

In [9]:
%%sql

DROP TABLE IF EXISTS ActivityTable;

CREATE TABLE ActivityTable (
    machine_id INTEGER NOT NULL,
    process_id INTEGER NOT NULL,
    activity_type status NOT NULL,
    timestamp FLOAT NOT NULL
);

INSERT INTO ActivityTable (machine_id, process_id, activity_type, timestamp)
VALUES
(0, 0, 'start', 0.712),
(0, 0, 'end', 1.520),
(0, 1, 'start', 3.140),
(0, 1, 'end', 4.120),
(1, 0, 'start', 0.550),
(1, 0, 'end', 1.550),
(1, 1, 'start', 0.430),
(1, 1, 'end', 1.420),
(2, 0, 'start', 4.100),
(2, 0, 'end', 4.512),
(2, 1, 'start', 2.500),
(2, 1, 'end', 5.000);

SELECT * FROM ActivityTable;

Running query in 'exercise_10'

12 rows affected.

12 rows affected.

machine_id,process_id,activity_type,timestamp
0,0,start,0.712
0,0,end,1.52
0,1,start,3.14
0,1,end,4.12
1,0,start,0.55
1,0,end,1.55
1,1,start,0.43
1,1,end,1.42
2,0,start,4.1
2,0,end,4.512


#### Solve the exercise

**Task.**
Find the average processing time `processing_time` for each machine **rounded to 3 decimal places** in **any order**

In [ ]:
%%sql
SELECT
    S.machine_id, ROUND(SUM(E.timestamp - S.timestamp)::numeric/COUNT(S.process_id), 3) AS processing_time
FROM
    ActivityTable S INNER JOIN ActivityTable E
ON
    S.machine_id = E.machine_id AND S.process_id = E.process_id
WHERE
    S.activity_type = 'start' AND E.activity_type = 'end'
GROUP BY
    S.machine_id;

Running query in 'exercise_10'

3 rows affected.

machine_id,processing_time
0,0.894
1,0.995
2,1.456


#### Explanation
- Perform a `SELF JOIN` on rows with same `machine_id` and `process_id` such that each row records the start time and end time of one process. This is done with `WHERE S.activity_type ='start' AND E.activity_type = 'end'`.
- Then for every `machine_id` take the average over `E.timestamp - S.timestamp`. This is done by `GROUP BY S.machine_id` and dividing by the `COUNT` over all `process_id` on one machine.
- Round the result to three decimals using `Round( _ , 3)`.

#### Close connection

In [11]:
%sql --close exercise_10
%sql postgresql+psycopg://postgres@localhost:5432/postgres
%sql DROP DATABASE IF EXISTS exercise_10;

Switching to connection 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

**Well done!** (approved by Leet)